In [4]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from numpy import pi
import numpy as np

In [5]:
def build_cuccaro_circuit(a, b):
    qreg_q = QuantumRegister(10, 'q')
    creg_c = ClassicalRegister(5, 'c')
    circuit = QuantumCircuit(qreg_q, creg_c)

    # q1,q4,q7,q10 = bits of A (LSB to MSB)
    # q2,q5,q8,q11 = bits of B (LSB to MSB)

    # encode A
    if a & 1: circuit.x(qreg_q[1])
    if a & 2: circuit.x(qreg_q[4])
    if a & 4: circuit.x(qreg_q[6])
    if a & 8: circuit.x(qreg_q[8])

    # encode B
    if b & 1: circuit.x(qreg_q[0])
    if b & 2: circuit.x(qreg_q[3])
    if b & 4: circuit.x(qreg_q[5])
    if b & 8: circuit.x(qreg_q[7])

    circuit.barrier()

    circuit.cx(qreg_q[6], qreg_q[5])
    circuit.cx(qreg_q[4], qreg_q[3])
    circuit.cx(qreg_q[8], qreg_q[7])
    circuit.cx(qreg_q[4], qreg_q[2])
    circuit.ccx(qreg_q[0], qreg_q[1], qreg_q[2])
    circuit.cx(qreg_q[6], qreg_q[4])
    circuit.ccx(qreg_q[2], qreg_q[3], qreg_q[4])
    circuit.cx(qreg_q[8], qreg_q[6])
    circuit.ccx(qreg_q[4], qreg_q[5], qreg_q[6])
    circuit.cx(qreg_q[8], qreg_q[9])
    circuit.barrier(qreg_q[3])
    circuit.x(qreg_q[3])
    circuit.ccx(qreg_q[6], qreg_q[7], qreg_q[9])
    circuit.x(qreg_q[5])
    circuit.cx(qreg_q[2], qreg_q[3])
    circuit.cx(qreg_q[4], qreg_q[5])
    circuit.cx(qreg_q[6], qreg_q[7])
    circuit.ccx(qreg_q[4], qreg_q[5], qreg_q[6])
    circuit.ccx(qreg_q[2], qreg_q[3], qreg_q[4])
    circuit.x(qreg_q[5])
    circuit.cx(qreg_q[8], qreg_q[6])
    circuit.ccx(qreg_q[0], qreg_q[1], qreg_q[2])
    circuit.x(qreg_q[3])
    circuit.cx(qreg_q[6], qreg_q[4])
    circuit.barrier(qreg_q[8])
    circuit.cx(qreg_q[4], qreg_q[2])
    circuit.barrier(qreg_q[0])
    circuit.barrier(qreg_q[6])
    circuit.barrier(qreg_q[8])
    circuit.cx(qreg_q[6], qreg_q[5])
    circuit.cx(qreg_q[8], qreg_q[7])
    circuit.cx(qreg_q[1], qreg_q[0])
    circuit.cx(qreg_q[4], qreg_q[3])
    circuit.measure(qreg_q[0], creg_c[0])
    circuit.measure(qreg_q[3], creg_c[1])
    circuit.measure(qreg_q[5], creg_c[2])
    circuit.measure(qreg_q[7], creg_c[3])
    circuit.measure(qreg_q[9], creg_c[4])


    return circuit

In [ ]:
# For research-grade accuracy, you usually want 10,000 shots, 
# but keeping it at 100 here for faster testing!
shots = 1000

# ====================================================
# 1. HARDWARE INSTRUCTION SET (BASIS GATES)
# ====================================================
# We must define the physical gates the quantum hardware actually uses
basis_gates = ['cx', 'id', 'rz', 'sx', 'x']

# ====================================================
# 2. PHASE NOISE MODEL (Z-ERROR)
# ====================================================
p = 0.0021

# 1-Qubit Phase Error
error_1 = pauli_error([
    ('Z', p),
    ('I', 1-p)
])

# 2-Qubit Phase Error (Z ⊗ Z)
error_2 = error_1.tensor(error_1)

noise_model = NoiseModel()

# Apply phase noise to the physical single-qubit gates
# The 'sx' gate is the crucial one—it creates the superposition!
noise_model.add_all_qubit_quantum_error(error_1, ['sx', 'x', 'id'])

# Apply phase noise to the physical two-qubit gates
noise_model.add_all_qubit_quantum_error(error_2, ['cx'])

# Bind the noise model to the simulator
sim = AerSimulator(noise_model=noise_model)

# ====================================================
# 3. NUMPY ARRAYS FOR METRICS
# ====================================================
ER_arr = np.zeros((16, 16))
NMED_arr = np.zeros((16, 16))
MRED_arr = np.zeros((16, 16))

# The maximum possible output for 4-bit + 4-bit is 30
D = 30

print("Running simulations, compiling circuits to basis gates...")

for a in range(16):
    for b in range(16):
        
        # Assume you have your build_vedral_circuit defined above
        qc = build_cuccaro_circuit(a, b)
        
        # CRITICAL STEP: Compile the high-level logic down to hardware basis gates
        compiled = transpile(qc, basis_gates=basis_gates)
        
        # Run the transpiled circuit
        counts = sim.run(
            compiled,
            shots=shots
        ).result().get_counts()

        correct = a + b
        # Ensure the binary string length matches your classical register size (e.g., 5 bits)
        correct_output = format(correct, '05b') 
        
        correct_counts = counts.get(correct_output, 0)
        
        # Error Rate calculation
        ER = 1 - correct_counts / shots
        
        total_ED = 0
        total_relative_ED = 0
        
        for output, freq in counts.items():
            noisy_decimal = int(output, 2)
            
            # Error Distance
            ED = abs(correct - noisy_decimal)
            total_ED += ED * freq
            
            if correct != 0:
                total_relative_ED += (ED / correct) * freq
                
        mean_ED = total_ED / shots
        
        NMED = mean_ED / D
        MRED = total_relative_ED / shots
        
        ER_arr[a, b] = ER
        NMED_arr[a, b] = NMED
        MRED_arr[a, b] = MRED

# ====================================================
# 4. OVERALL METRICS
# ====================================================
er = np.mean(ER_arr)
nmed = np.mean(NMED_arr)
mred = np.mean(MRED_arr)

print("\n--- Phase Noise Results ---")
print(f"ER (%) : {er * 100:.4f}")
print(f"NMED   : {nmed:.6f}")
print(f"MRED   : {mred:.6f}")

Running simulations, compiling circuits to basis gates...

--- Phase Noise Results ---
ER (%) : 6.1875
NMED   : 0.011099
MRED   : 0.032762
